<a href="https://colab.research.google.com/github/gocenalper/BigramModels/blob/main/make_more.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

In [ ]:
from pathlib import Path
import urllib.request

In [ ]:
_BASE_DIR = Path(__file__).parent if "__file__" in globals() else Path.cwd()
DATA_PATH = _BASE_DIR / "data" / "names.txt"
DATA_URL = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"

# Create the data directory if it doesn't exist
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

# Download the file if it doesn't exist locally
if not DATA_PATH.exists():
    print(f"Downloading {DATA_URL} to {DATA_PATH}...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print("Download complete.")

In [ ]:
words = DATA_PATH.read_text(encoding="utf-8").splitlines()

words[:10]

In [ ]:
len(words)

In [ ]:
b = {}
for w in words:
  chs = ['<S>'] + list(w) + ['<E>']
  for ch1, ch2 in zip(chs, chs[1:]):
    bigram = (ch1, ch2)
    b[bigram] = b.get(bigram, 0) + 1

In [ ]:
chars = sorted(b.items(), key = lambda kv: -kv[1])

In [ ]:
N = torch.zeros((27, 27), dtype=torch.int32)

In [ ]:
chars = sorted(list(set(''.join(words))))

stoi = {ch: i+1 for i, ch in enumerate(chars)}
stoi['.'] = 0

itos = {i: ch for ch, i in stoi.items()}

In [ ]:
for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    N[ix1, ix2] += 1

In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline
plt.figure(figsize=(16, 16))
plt.imshow(N, cmap='Blues')

for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off')

In [ ]:
p = N[0].float()
p = p/p.sum()
p

In [ ]:
g = torch.Generator().manual_seed(2147483647)

for _ in range(10):
  out = []
  ix = 0
  while True:
    p = N[ix].float()
    p = p/p.sum()
    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    out.append(itos[ix])
    if ix == 0:
      break
  print(''.join(out))

### Creating the Probability Matrix (P) with Broadcasting
Here we use PyTorch's **broadcasting** rules. `P.sum(1, keepdim=True)` returns a `27x1` vector. `P` is `27x27`. PyTorch matches dimensions from right to left, so this `27x1` matrix is automatically copied (broadcast) across the entire `27x27` matrix during the division operation.

Additionally, to prevent probabilities of `0` (and their logarithms from becoming negative infinity) in the future, we add `+1` to all counts (Laplace Smoothing).

In [ ]:
# We perform model smoothing by adding 1
P = (N+1).float()

# We normalize the entire matrix at once using broadcasting:
# P = 27x27
# P.sum(1, keepdim=True) = 27x1
P /= P.sum(1, keepdim=True)

print(f"P shape: {P}")

We are updating our sampling loop with the `P` matrix we created. Now we no longer need to calculate `.sum()` in each iteration of the loop, making the code much more performant:

In [ ]:
g = torch.Generator().manual_seed(2147483647)

for _ in range(10):
  out = []
  ix = 0
  while True:
    # Now we directly retrieve the relevant row from the P matrix
    p = P[ix]

    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    out.append(itos[ix])
    if ix == 0:
      break
  print(''.join(out))

### Measuring Model Performance: Negative Log-Likelihood (NLL)

Now we can generate words from our model. But **how good** is this model?
To measure this, we take the words in our dataset and see how much probability our model assigns to the character transitions (bigrams) within these words.

A good model should assign high probabilities to the actual transitions in the dataset.
- The product of probabilities is called **Likelihood**.
- Since probabilities are between 0 and 1, multiplying them together makes the number very small. Therefore, we use **Log-Likelihood** (the sum of the logarithms of the probabilities).
- In machine learning, we want to **minimize** the loss value, so we multiply this value by minus one to obtain **Negative Log-Likelihood (NLL)**. The lower the average NLL, the better our model!

In [ ]:
# Let's look at the probabilities and log-probabilities of the first 3 words as an example:
log_likelihood = 0.0
n = 0

for w in words:
  print(f"--- {w} ---")
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    prob = P[ix1, ix2]
    logprob = torch.log(prob)
    log_likelihood += logprob
    n += 1
    #print(f'{ch1}{ch2}: prob={prob:.4f}, log-prob={logprob:.4f}')

#print('='*20)
#print(f'log_likelihood = {log_likelihood:.4f}')
nll = -log_likelihood
#print(f'negative log_likelihood = {nll:.4f}')
print(f'Average Loss (Loss / NLL) = {nll/n:.4f}')

Now let's perform this operation for our **entire dataset (words)** and see the overall performance (loss value) of our model.

In Karpathy's makemore series, this is our fundamental loss function. Throughout the training, our goal will be to reduce this average NLL value (which will be around 2.45) by building more advanced neural network models.

In [ ]:
# Calculate NLL (Loss) for the entire dataset
log_likelihood = 0.0
n = 0

for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    prob = P[ix1, ix2]
    logprob = torch.log(prob)
    log_likelihood += logprob
    n += 1

nll = -log_likelihood
print(f'Average Loss (Loss/NLL) for the entire dataset = {nll/n:.4f}')

### Why was Model Smoothing Necessary?

Following the video's sequence, after calculating NLL, we need to see what happens if we test our model with a character sequence it has never seen before. For example, the word **"andrejq"**.

In our dataset, 'q' never follows 'j'. If we hadn't added `+1` when creating the matrix, the probability of this transition would have been `0`, and its logarithm would be `-infinity`, causing the model's loss to explode.

Since we previously added at least 1 count to each transition by setting `P = (N+1).float()` (this is called **Laplace Smoothing**), we avoided this infinite loss error. Let's see:

In [ ]:
# Let's test the word "andrejq"
log_likelihood = 0.0
n = 0
for w in ["andrejq"]:
  print(f"--- {w} ---")
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    prob = P[ix1, ix2]
    logprob = torch.log(prob)
    log_likelihood += logprob
    n += 1
    print(f'{ch1}{ch2}: prob={prob:.4f}, log-prob={logprob:.4f}')

nll = -log_likelihood
print(f'Average Loss = {nll/n:.4f}')
print("\nPay attention to the 'jq' transition! Its probability is not 0; thanks to +1 smoothing, it has a very small probability, and the loss does not become infinite.")

### Section 2: Neural Network Approach

We have finished our statistical (count-based) Bigram model and smoothed it. In Karpathy's video, we are now moving to a brand new chapter: we will solve the same problem with a **Neural Network**!

For this, we first need to prepare our dataset for training the network. Our inputs (`x`) will be the first character, and our targets (`y`) will be the second character that should follow it.

In [ ]:
# Create the training set for the neural network
xs, ys = [], []

for w in words: # Create data from all words
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    xs.append(ix1)
    ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)

print(f"Total {len(xs)} examples created.")
print(f"Inputs (xs) shape: {xs.shape}")
print(f"Targets (ys) shape: {ys.shape}")

### One-Hot Encoding
To remove the meaningless magnitude relationship between the inputs (characters) we feed to the neural network, we convert them into 27-dimensional vectors (consisting of 0s and 1s).

In [ ]:
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

# Convert xs tensor to one-hot vectors
# num_classes=27 because we have 27 different characters
xenc = F.one_hot(xs, num_classes=27).float()

print("xenc shape:", xenc.shape)
print("xenc data type:", xenc.dtype)

# Visualization (Yellow dots represent 1s, purple areas represent 0s)
# Displaying only the first 5 examples for practical visualization
plt.imshow(xenc[:5])

The shape of `xenc` is `(228146, 27)` because:

*   **`228146`**: This is the total number of bigram examples created from all the words in your dataset (`len(xs)`). Each row in `xenc` represents one of these input characters.
*   **`27`**: This is the `num_classes` parameter used in `F.one_hot()`. It corresponds to the total number of unique characters in your vocabulary (26 letters of the alphabet + the special `.` character). Each input character from `xs` is transformed into a 27-dimensional vector, with a `1` at the index corresponding to the character and `0`s elsewhere (one-hot encoding).

### First Layer of the Neural Network: Weights and Logits

We create a 27x27 weight matrix (`W`) initialized with random numbers. This will be our most basic (single-layer) neural network.

When we multiply our inputs (`xenc`) by the weights (`@` operator), we will obtain the outputs generated by 27 different neurons for each input character.

In [ ]:
# Initialize the weight matrix with random numbers (Normal distribution)
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g)

# Forward pass: Inputs x Weights
# (5, 27) @ (27, 27) = (5, 27)
logits = xenc @ W

print("Weight matrix (W) shape:", W.shape)
print("Output (logits) matrix shape:", logits.shape)
print("\nLogits for our first example (first character):\n", logits[0])

### From Logits to Probabilities (Softmax)

You've hit on a very good point! However, there's a small but critical detail: the results from `xenc @ W` are currently **not probabilities**, but just the **raw scores (logits)** generated by the model. Some are negative, some are positive.

Since our inputs (`xenc`) are One-Hot encoded (only one element is 1, the rest are 0), the `xenc @ W` operation is essentially **pulling out** (a kind of filtering) the row corresponding to the relevant character from the `W` matrix.

For the model to learn:
1. We need to convert these raw scores (logits) into **probabilities (between 0 and 1)**.
2. We will compare the obtained probabilities with the actual character to find our error.
3. We will go backward and **update the weights in the W matrix** (Backpropagation).

Now let's convert these raw scores into probabilities. This process is called **Softmax** in neural networks:

In [ ]:
# Step 1: We take the exponential of the scores to make them positive (exp)
# This step actually creates an effect similar to the 'N' (counts) matrix in the statistical model.
counts = logits.exp()

# Step 2: We divide each row by its sum to turn them into probabilities (between 0-1 and sum to 1)
# This step is similar to creating the 'P' matrix (normalization) in the statistical model.
probs = counts / counts.sum(1, keepdims=True)

print("Probability matrix shape (probs):", probs.shape)
print("\nProbabilities for our first example:\n", probs[0])
print("\nThe sum of these probabilities (should be 1.0):", probs[0].sum().item())

### Calculating the Loss Value

Now we have the probability of 27 different characters appearing for each input character. So, how much probability did our model assign to the correct characters?

Our correct characters (targets) are in the `ys` tensor: `[5, 13, 13, 1, 0]`
- For the 0th input ('.'), the 5th character ('e') should appear.
- For the 1st input ('e'), the 13th character ('m') should appear.

Let's extract (pluck) the probabilities corresponding to these correct characters from the `probs` matrix and calculate the Negative Log-Likelihood (NLL).

In [ ]:
# We have len(xs) examples
num_examples = len(xs)

# We extract the probabilities of the correct characters using PyTorch's indexing capabilities
# e.g., probs[0, 5], probs[1, 13], probs[2, 13] ...
correct_probs = probs[torch.arange(num_examples), ys]
#print("Probabilities assigned to correct characters:", correct_probs) # This output can be very long for the full dataset

# Take their logarithms
log_probs = torch.log(correct_probs)

# Calculate the Loss value by taking the negative mean (NLL)
loss = -log_probs.mean()
print(f"\nOur Loss value: {loss.item():.4f}")


In [ ]:
# Initialize weights W with requires_grad=True for gradient calculation
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

# --- Training Loop (One Step) ---

# 1. Forward pass
logits = xenc @ W # Predict log-counts
counts = logits.exp() # Equivalent to N
probs = counts / counts.sum(1, keepdims=True) # Probabilities for next character

# 2. Calculate loss
# Negative Log Likelihood
loss = -probs[torch.arange(len(xs)), ys].log().mean()

print(f"Loss before optimization: {loss.item():.4f}")

# 3. Backward pass (compute gradients)
W.grad = None # Reset gradients
loss.backward()

# 4. Update weights
# We multiply by a small learning rate (e.g., -0.1) to descend the gradient
W.data += -0.1 * W.grad

# 5. Recalculate loss after update
logits = xenc @ W
counts = logits.exp()
probs = counts / counts.sum(1, keepdims=True)
loss = -probs[torch.arange(len(xs)), ys].log().mean()

print(f"Loss after one optimization step: {loss.item():.4f}")

In [ ]:
import wandb
wandb.login()

wandb.init(project="bigram_test", name="nn_solution")

In [ ]:
# Initialize weights W with requires_grad=True for gradient calculation
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

# --- Full Training Loop ---
learning_rate = 10
num_iterations = 10000

print("Starting training...")
for k in range(num_iterations):

  # 1. Forward pass
  logits = xenc @ W # Predict log-counts
  counts = logits.exp() # Equivalent to N
  probs = counts / counts.sum(1, keepdims=True) # Probabilities for next character

  # 2. Calculate loss (Negative Log Likelihood)
  loss = -probs[torch.arange(len(xs)), ys].log().mean()

  # 3. Backward pass (compute gradients)
  W.grad = None # Reset gradients to zero
  loss.backward()

  # 4. Update weights
  W.data += -learning_rate * W.grad

  if k % 10 == 0: # Print loss every 10 iterations
    print(f"Iteration {k+1}/{num_iterations}: Loss = {loss.item():.4f}")

  wandb.log({"loss": loss.item(), "step": k})

print(f"Training finished. Final Loss = {loss.item():.4f}")

wandb.finish()

### Sampling from the Neural Network Model

Now that our neural network has been trained, let's use it to generate new names. The process is similar to our statistical bigram model: we start with a special character ('.') and then repeatedly sample the next character based on the probabilities generated by the neural network, until we generate another '.' (end-of-word token).

In [ ]:
g = torch.Generator().manual_seed(2147483647 + 10) # Using a different seed for sampling

for _ in range(20):
  out = []
  ix = 0
  while True:
    # Forward pass for a single character (ix)
    xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
    logits = xenc @ W # W is the trained weight matrix
    counts = logits.exp()
    p = counts / counts.sum(1, keepdims=True) # Probabilities for the next character

    # Sample from the distribution
    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    out.append(itos[ix])
    if ix == 0:
      break
  print(''.join(out))